# 导出 ONNX 并发布到 Hugging Face

这个 notebook 只做三件事：导出 ONNX、创建 Model 仓库、上传 Static Space。
不需要 GPU，选 Colab CPU 内核即可。

使用前确认：
1. Drive 的 `MyDrive/alexnet_cifar100/best_model.pth` 还在
2. 准备好 Hugging Face 的 Write token
3. 不要运行 `dev.ipynb` 的训练 cell

In [1]:
import torch
print(torch.__version__)
print("cuda:", torch.cuda.is_available())

2.14.1+cpu
cuda: False


## 1. 挂载 Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. 拉取最新代码

In [3]:
import os

if not os.path.exists('/content/alexnet-cifar100'):
    !git clone https://github.com/Wait4theWind/alexnet-cifar100.git

%cd /content/alexnet-cifar100
!git pull

/content/alexnet-cifar100
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 2.94 KiB | 1004.00 KiB/s, done.
From https://github.com/Wait4theWind/alexnet-cifar100
   b31ebf5..bb35493  main       -> origin/main
Updating b31ebf5..bb35493
Fast-forward
 README.md          |   2 +-
 export_space.ipynb | 103 +++++++++++++++++++++++++++++++++++++++++------------
 2 files changed, 81 insertions(+), 24 deletions(-)


In [4]:
!pip install -q 'huggingface_hub>=0.25,<2.0' onnx

## 3. 导出 ONNX

In [5]:
!python src/export_onnx.py --weights /content/drive/MyDrive/alexnet_cifar100/best_model.pth --out space_static/alexnet_cifar100.onnx
!ls -lh space_static/alexnet_cifar100.onnx
!cp space_static/alexnet_cifar100.onnx /content/drive/MyDrive/alexnet_cifar100/

/content/alexnet-cifar100/src/export_onnx.py:28: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/torchscript_exporter/onnx_proto_utils.py", line 183, in _add_onnxscript_fn
    import onnx
ModuleNotFoundError: No module named 'onnx'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/content/alexnet-cifar100/src/export_onnx.py", line 53, in <module>
    main()
    ~~~~^^
  File "/content/alexnet-cifar100/src/export_onnx.py", line 28, in main
    torch.onnx.export(
    ~~~~~~~~

## 4. 登录 Hugging Face

运行下面的 cell 后粘贴 token。token 在 https://huggingface.co/settings/tokens 创建，类型选 Write。

In [ ]:
from huggingface_hub import login
login()

## 5. 创建仓库并上传

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.create_repo("alexnet-cifar100", repo_type="model", exist_ok=True)
api.create_repo("alexnet-cifar100-demo", repo_type="space", space_sdk="static", exist_ok=True)

api.upload_file(
    path_or_fileobj="/content/drive/MyDrive/alexnet_cifar100/best_model.pth",
    path_in_repo="best_model.pth",
    repo_id="bbbegemot/alexnet-cifar100",
    repo_type="model",
)
api.upload_file("src/model.py", "model.py", repo_id="bbbegemot/alexnet-cifar100", repo_type="model")
api.upload_file("model_card/README.md", "README.md", repo_id="bbbegemot/alexnet-cifar100", repo_type="model")

api.upload_folder(
    folder_path="space_static",
    repo_id="bbbegemot/alexnet-cifar100-demo",
    repo_type="space",
)

print("上传完成")

## 6. 测试

- Model: https://huggingface.co/bbbegemot/alexnet-cifar100
- Space: https://huggingface.co/spaces/bbbegemot/alexnet-cifar100-demo

Space 里上传图片，等模型加载完成后点“开始预测”。